In [3]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

In [4]:
def step_env(state, action):
    noise = np.random.normal(0, 0.05)          # aleatoric noise
    next_state = state + action + noise
    reward = -(next_state ** 2)                # closer to 0 = better
    return next_state, reward

In [5]:
class ProbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 2)   # outputs [mean, logvar]
        )

    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        out = self.net(x)
        mean, logvar = out[:, 0:1], out[:, 1:2]
        return mean, logvar

    def loss(self, s, a, s_next):
        mean, logvar = self.forward(s, a)
        var = torch.exp(logvar)
        # Gaussian negative log‑likelihood
        return (0.5 * ((s_next - mean) ** 2 / var + logvar)).mean()

In [6]:
N_MODELS = 3
models = [ProbNet() for _ in range(N_MODELS)]
optims = [torch.optim.Adam(m.parameters(), lr=1e-2) for m in models]

def train_ensemble(S, A, S2, epochs=100):
    n = len(S)
    for m, opt in zip(models, optims):
        idx = np.random.randint(0, n, n)       # bootstrap sample
        s, a, s2 = S[idx], A[idx], S2[idx]
        s_t = torch.tensor(s, dtype=torch.float32).unsqueeze(1)
        a_t = torch.tensor(a, dtype=torch.float32).unsqueeze(1)
        s2_t = torch.tensor(s2, dtype=torch.float32).unsqueeze(1)
        for _ in range(epochs):
            opt.zero_grad()
            loss = m.loss(s_t, a_t, s2_t)
            loss.backward()
            opt.step()

In [7]:
def imagine_next_state(state, action):
    m = models[np.random.randint(N_MODELS)]
    s_t = torch.tensor([[state]], dtype=torch.float32)
    a_t = torch.tensor([[action]], dtype=torch.float32)
    with torch.no_grad():
        mean, logvar = m(s_t, a_t)
        std = torch.exp(0.5 * logvar)
        sample = mean + std * torch.randn_like(mean)
    return sample.item()

In [8]:
def cem_plan(state, horizon=5, n_candidates=100, n_elites=10, n_iters=3):
    mean = np.zeros(horizon)
    std = np.ones(horizon)
    for _ in range(n_iters):
        candidates = np.random.normal(mean, std, size=(n_candidates, horizon))
        rewards = np.zeros(n_candidates)
        for c in range(n_candidates):
            s = state
            total_r = 0
            for t in range(horizon):
                s = imagine_next_state(s, candidates[c, t])
                total_r += -(s ** 2)
            rewards[c] = total_r
        elite_idx = rewards.argsort()[-n_elites:]
        elites = candidates[elite_idx]
        mean, std = elites.mean(axis=0), elites.std(axis=0) + 1e-3
    return mean[0]

In [9]:
state = np.random.uniform(-5, 5)
S, A, S2 = [], [], []
for _ in range(200):
    a = np.random.uniform(-1, 1)
    s2, _ = step_env(state, a)
    S.append(state); A.append(a); S2.append(s2)
    state = s2

S, A, S2 = np.array(S), np.array(A), np.array(S2)
train_ensemble(S, A, S2)

In [10]:
state = 5.0
for t in range(15):
    action = cem_plan(state)
    next_state, reward = step_env(state, action)
    S = np.append(S, state)
    A = np.append(A, action)
    S2 = np.append(S2, next_state)
    print(f"t={t:2d} state={state:6.2f} action={action:6.2f} reward={reward:6.2f}")
    state = next_state
    train_ensemble(S, A, S2)

t= 0 state=  5.00 action= -2.94 reward= -4.10
t= 1 state=  2.03 action= -2.00 reward= -0.01
t= 2 state=  0.08 action= -0.12 reward= -0.01
t= 3 state= -0.08 action=  0.05 reward= -0.02
t= 4 state= -0.14 action=  0.12 reward= -0.01
t= 5 state= -0.11 action=  0.02 reward= -0.00
t= 6 state= -0.02 action= -0.02 reward= -0.00
t= 7 state=  0.06 action=  0.12 reward= -0.06
t= 8 state=  0.25 action= -0.11 reward= -0.04
t= 9 state=  0.19 action= -0.10 reward= -0.01
t=10 state=  0.12 action= -0.17 reward= -0.00
t=11 state= -0.06 action=  0.12 reward= -0.01
t=12 state=  0.09 action=  0.01 reward= -0.01
t=13 state=  0.10 action= -0.13 reward= -0.00
t=14 state= -0.05 action=  0.06 reward= -0.00
